# METR-LA Traffic Prediction — Final Clean Pipeline

**Final Test:** 50/50 Extra Trees + HGB → **R² = 0.8358**


In [ ]:

import gc
import pickle
import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('Imports ready.')


Imports ready.


In [ ]:

metr = pd.read_hdf('../../dataset/METR-LA/metr-la.h5')
metr.index = pd.to_datetime(metr.index)
metr.index.name = 'time'

weather = pd.read_csv('../../dataset/LA_weather_2012.csv')
weather['time'] = pd.to_datetime(weather['time'])

holiday = pd.read_csv('../../dataset/USHolidayDataset/USHolidayDates.csv')
holiday['Date'] = pd.to_datetime(holiday['Date'])
holiday = holiday[['Date', 'Holiday']]

events = pd.read_csv('../../dataset/USHolidayDataset/Whats_Happening_LA.csv')
events['Event Date & Time Start'] = pd.to_datetime(
    events['Event Date & Time Start'],
    errors='coerce'
)

print('METR-LA:', metr.shape)
print('Weather:', weather.shape)
print('Holidays:', holiday.shape)
print('Events:', events.shape)


C:\Users\alisaleh\AppData\Local\Temp\ipykernel_496\3159730942.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  events['Event Date & Time Start'] = pd.to_datetime(


METR-LA: (34272, 207)
Weather: (2856, 5)
Holidays: (342, 2)
Events: (29519, 20)


In [ ]:
metr_reset = metr.reset_index().sort_values('time')
weather = weather.sort_values('time')

merged = pd.merge_asof(
    metr_reset,
    weather,
    on='time',
    direction='backward'
)

merged['Date'] = merged['time'].dt.normalize()

merged = merged.merge(
    holiday,
    on='Date',
    how='left'
)

merged['Holiday'] = merged['Holiday'].fillna('None')

print('Merged shape:', merged.shape)

del metr_reset, weather, holiday, metr
gc.collect()


Merged shape: (34272, 214)


23

## 4. RAM-Friendly Feature Pipeline

In [ ]:
import gc
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder


df = merged

df['time'] = pd.to_datetime(
    df['time']
)

df = df.sort_values(
    'time'
).reset_index(
    drop=True
)


sensor_columns = [
    col
    for col in df.columns
    if str(col).isdigit()
]

N_SENSORS = len(sensor_columns)
N_TIMESTAMPS = len(df)


print(
    'Number of sensors:',
    N_SENSORS
)

print(
    'Number of timestamps:',
    N_TIMESTAMPS
)



speed_matrix = (
    df[sensor_columns]
    .to_numpy(
        dtype=np.float32,
        copy=True
    )
)


print(
    'Speed matrix:',
    speed_matrix.shape
)

print(
    'Speed matrix RAM:',
    round(
        speed_matrix.nbytes / 1024**2,
        2
    ),
    'MB'
)


times = df['time']

hour = (
    times.dt.hour
    .to_numpy(
        dtype=np.int8
    )
)

minute = (
    times.dt.minute
    .to_numpy(
        dtype=np.int8
    )
)

dow = (
    times.dt.dayofweek
    .to_numpy(
        dtype=np.int8
    )
)

month = (
    times.dt.month
    .to_numpy(
        dtype=np.int8
    )
)


is_weekend = (
    dow >= 5
).astype(
    np.int8
)


is_holiday = (
    df['Holiday']
    .ne('None')
    .to_numpy(
        dtype=np.int8
    )
)


hour_sin = np.sin(
    2 * np.pi * hour / 24
).astype(
    np.float32
)

hour_cos = np.cos(
    2 * np.pi * hour / 24
).astype(
    np.float32
)

dow_sin = np.sin(
    2 * np.pi * dow / 7
).astype(
    np.float32
)

dow_cos = np.cos(
    2 * np.pi * dow / 7
).astype(
    np.float32
)

month_sin = np.sin(
    2 * np.pi * month / 12
).astype(
    np.float32
)

month_cos = np.cos(
    2 * np.pi * month / 12
).astype(
    np.float32
)


temperature = (
    df['temperature_2m']
    .to_numpy(
        dtype=np.float32
    )
)

humidity = (
    df['relative_humidity_2m']
    .to_numpy(
        dtype=np.float32
    )
)

rain = (
    df['rain']
    .to_numpy(
        dtype=np.float32
    )
)

weather_code = (
    df['weather_code']
    .to_numpy(
        dtype=np.float32
    )
)


temp_humidity = (
    temperature
    *
    humidity
).astype(
    np.float32
)


rain_flag = (
    rain > 0
).astype(
    np.int8
)


rain_intensity = np.log1p(
    np.maximum(
        rain,
        0
    )
).astype(
    np.float32
)


bad_weather = (
    (rain > 0)
    |
    (weather_code >= 50)
).astype(
    np.int8
)


morning_rush = (
    (hour >= 6)
    &
    (hour <= 9)
).astype(
    np.int8
)


evening_rush = (
    (hour >= 15)
    &
    (hour <= 19)
).astype(
    np.int8
)


rush_hour = (
    (morning_rush == 1)
    |
    (evening_rush == 1)
).astype(
    np.int8
)


events_data = events

event_start = pd.to_datetime(
    events_data[
        'Event Date & Time Start'
    ],
    errors='coerce'
)


event_start = (
    event_start
    .dropna()
)


event_hour_counts = (
    event_start
    .dt.floor('h')
    .value_counts()
)


timestamp_hours = (
    times.dt.floor('h')
)


event_count_time = (
    timestamp_hours
    .map(
        event_hour_counts
    )
    .fillna(0)
    .to_numpy(
        dtype=np.float32
    )
)


has_event_time = (
    event_count_time > 0
).astype(
    np.int8
)


print(
    'Valid events:',
    len(event_start)
)


START = 24
END = N_TIMESTAMPS - 3

valid_times = np.arange(
    START,
    END
)

N_VALID = len(
    valid_times
)


print(
    'Valid timestamps:',
    N_VALID
)


target_matrix = (
    speed_matrix[
        valid_times + 3
    ]
)



sensor_ids = np.array(
    [
        int(x)
        for x in sensor_columns
    ],
    dtype=np.int32
)



sensor_encoded = np.arange(
    N_SENSORS,
    dtype=np.int16
)


feature_names = [

    'sensor_id_encoded',

    'speed',

    'lag_1',
    'lag_2',
    'lag_3',
    'lag_4',
    'lag_6',
    'lag_9',
    'lag_12',
    'lag_18',
    'lag_24',

    'rolling_mean_3',
    'rolling_mean_6',
    'rolling_mean_12',
    'rolling_mean_24',

    'rolling_std_6',
    'rolling_std_12',
    'rolling_std_24',

    'speed_change_1',
    'speed_change_3',
    'speed_change_6',

    'speed_vs_mean_6',
    'speed_vs_mean_12',

    'temperature_2m',
    'relative_humidity_2m',
    'rain',
    'weather_code',

    'temp_humidity_interaction',
    'rain_flag',
    'rain_intensity',
    'bad_weather',

    'hour',
    'minute',
    'day_of_week',
    'month',

    'is_weekend',
    'is_holiday',

    'is_morning_rush',
    'is_evening_rush',
    'is_rush_hour',

    'hour_sin',
    'hour_cos',
    'dow_sin',
    'dow_cos',
    'month_sin',
    'month_cos',

    'event_count',
    'has_event'
]


N_FEATURES = len(
    feature_names
)


print(
    'Number of features:',
    N_FEATURES
)


split_position = int(
    N_VALID * 0.8
)


train_time_positions = (
    valid_times[
        :split_position
    ]
)

test_time_positions = (
    valid_times[
        split_position:
    ]
)


print(
    'Train timestamps:',
    len(
        train_time_positions
    )
)

print(
    'Test timestamps:',
    len(
        test_time_positions
    )
)


def build_features_for_times(
    time_positions
):

    n_t = len(
        time_positions
    )

    n_rows = (
        n_t
        *
        N_SENSORS
    )


    X = np.empty(
        (
            n_rows,
            N_FEATURES
        ),
        dtype=np.float32
    )


    
    row = 0


    X[:, 0] = np.tile(
        sensor_encoded,
        n_t
    )


    X[:, 1] = speed_matrix[
        time_positions
    ].reshape(
        -1
    )


    lag_values = [
        1,
        2,
        3,
        4,
        6,
        9,
        12,
        18,
        24
    ]


    col = 2


    for lag in lag_values:

        X[:, col] = (
            speed_matrix[
                time_positions - lag
            ]
            .reshape(-1)
        )

        col += 1


    speed_df = pd.DataFrame(
        speed_matrix
    )


    shifted = (
        speed_df
        .shift(1)
    )


    rolling_windows = [
        3,
        6,
        12,
        24
    ]


    # Rolling means
    for window in rolling_windows:

        rolling_mean = (
            shifted
            .rolling(
                window,
                min_periods=window
            )
            .mean()
            .to_numpy(
                dtype=np.float32
            )
        )

        X[:, col] = (
            rolling_mean[
                time_positions
            ]
            .reshape(-1)
        )

        col += 1

        del rolling_mean


    for window in [
        6,
        12,
        24
    ]:

        rolling_std = (
            shifted
            .rolling(
                window,
                min_periods=window
            )
            .std()
            .to_numpy(
                dtype=np.float32
            )
        )

        X[:, col] = (
            rolling_std[
                time_positions
            ]
            .reshape(-1)
        )

        col += 1

        del rolling_std


    del shifted
    del speed_df

    gc.collect()


   
    current = speed_matrix[
        time_positions
    ]

    lag1 = speed_matrix[
        time_positions - 1
    ]

    lag3 = speed_matrix[
        time_positions - 3
    ]

    lag6 = speed_matrix[
        time_positions - 6
    ]


    X[:, col] = (
        current - lag1
    ).reshape(-1)

    col += 1


    X[:, col] = (
        current - lag3
    ).reshape(-1)

    col += 1


    X[:, col] = (
        current - lag6
    ).reshape(-1)

    col += 1


  
    speed_df = pd.DataFrame(
        speed_matrix
    )

    shifted = speed_df.shift(1)


    rolling6 = (
        shifted
        .rolling(
            6,
            min_periods=6
        )
        .mean()
        .to_numpy(
            dtype=np.float32
        )
    )


    rolling12 = (
        shifted
        .rolling(
            12,
            min_periods=12
        )
        .mean()
        .to_numpy(
            dtype=np.float32
        )
    )


    X[:, col] = (
        current
        /
        (
            rolling6[
                time_positions
            ]
            + 1e-6
        )
    ).reshape(-1)

    col += 1


    X[:, col] = (
        current
        /
        (
            rolling12[
                time_positions
            ]
            + 1e-6
        )
    ).reshape(-1)

    col += 1


    del rolling6
    del rolling12
    del shifted
    del speed_df

    gc.collect()


    X[:, col] = np.repeat(
        temperature[
            time_positions
        ],
        N_SENSORS
    )

    col += 1


    X[:, col] = np.repeat(
        humidity[
            time_positions
        ],
        N_SENSORS
    )

    col += 1


    X[:, col] = np.repeat(
        rain[
            time_positions
        ],
        N_SENSORS
    )

    col += 1


    X[:, col] = np.repeat(
        weather_code[
            time_positions
        ],
        N_SENSORS
    )

    col += 1


    X[:, col] = np.repeat(
        temp_humidity[
            time_positions
        ],
        N_SENSORS
    )

    col += 1


    X[:, col] = np.repeat(
        rain_flag[
            time_positions
        ],
        N_SENSORS
    )

    col += 1


    X[:, col] = np.repeat(
        rain_intensity[
            time_positions
        ],
        N_SENSORS
    )

    col += 1


    X[:, col] = np.repeat(
        bad_weather[
            time_positions
        ],
        N_SENSORS
    )

    col += 1


  
    time_arrays = [
        hour,
        minute,
        dow,
        month,
        is_weekend,
        is_holiday,
        morning_rush,
        evening_rush,
        rush_hour,
        hour_sin,
        hour_cos,
        dow_sin,
        dow_cos,
        month_sin,
        month_cos
    ]


    for arr in time_arrays:

        X[:, col] = np.repeat(
            arr[
                time_positions
            ],
            N_SENSORS
        )

        col += 1


    X[:, col] = np.repeat(
        event_count_time[
            time_positions
        ],
        N_SENSORS
    )

    col += 1


    X[:, col] = np.repeat(
        has_event_time[
            time_positions
        ],
        N_SENSORS
    )

    col += 1


    print(
        'Features built:',
        X.shape
    )

    print(
        'RAM:',
        round(
            X.nbytes / 1024**2,
            2
        ),
        'MB'
    )


    return X

def build_target(
    time_positions
):

    return (
        speed_matrix[
            time_positions + 3
        ]
        .reshape(-1)
        .astype(
            np.float32
        )
    )



TRAIN_SAMPLES = 300_000


rng = np.random.default_rng(
    42
)


total_train_rows = (
    len(train_time_positions)
    *
    N_SENSORS
)


sample_size = min(
    TRAIN_SAMPLES,
    total_train_rows
)


sample_indices = (
    rng.choice(
        total_train_rows,
        size=sample_size,
        replace=False
    )
)


sample_time_idx = (
    sample_indices
    // N_SENSORS
)

sample_sensor_idx = (
    sample_indices
    % N_SENSORS
)


sample_times = (
    train_time_positions[
        sample_time_idx
    ]
)


print()
print(
    'Training samples:',
    sample_size
)


X_train_full = build_features_for_times(
    sample_times
)


row_indices = (
    sample_time_idx
    *
    N_SENSORS
    +
    sample_sensor_idx
)


X_train = (
    X_train_full[
        row_indices
    ]
)


y_train = (
    speed_matrix[
        sample_times + 3,
        sample_sensor_idx
    ]
    .astype(
        np.float32
    )
)


del X_train_full
del row_indices
del sample_indices
del sample_time_idx
del sample_sensor_idx
del sample_times

gc.collect()


print(
    'FINAL X_train:',
    X_train.shape
)

print(
    'FINAL y_train:',
    y_train.shape
)

print(
    'X_train RAM:',
    round(
        X_train.nbytes / 1024**2,
        2
    ),
    'MB'
)



print()
print(
    'Building test features...'
)


X_test = build_features_for_times(
    test_time_positions
)


y_test = build_target(
    test_time_positions
)


print(
    'X_test:',
    X_test.shape
)

print(
    'y_test:',
    y_test.shape
)



del df

gc.collect()



print()
print(
    '=' * 55
)

print(
    'RAM-FRIENDLY DATASET READY'
)

print(
    '=' * 55
)

print(
    'Features:',
    N_FEATURES
)

print(
    'X_train:',
    X_train.shape
)

print(
    'y_train:',
    y_train.shape
)

print(
    'X_test:',
    X_test.shape
)

print(
    'y_test:',
    y_test.shape
)

print(
    'X_train dtype:',
    X_train.dtype
)

print(
    'X_test dtype:',
    X_test.dtype
)

print(
    'X_train RAM:',
    round(
        X_train.nbytes / 1024**2,
        2
    ),
    'MB'
)

print(
    'X_test RAM:',
    round(
        X_test.nbytes / 1024**2,
        2
    ),
    'MB'
)

print(
    '=' * 55
)

Number of sensors: 207
Number of timestamps: 34272
Speed matrix: (34272, 207)
Speed matrix RAM: 27.06 MB
Valid events: 29478
Valid timestamps: 34245
Number of features: 48
Train timestamps: 27396
Test timestamps: 6849

Training samples: 300000
Features built: (62100000, 48)
RAM: 11370.85 MB
FINAL X_train: (300000, 48)
FINAL y_train: (300000,)
X_train RAM: 54.93 MB

Building test features...
Features built: (1417743, 48)
RAM: 259.6 MB
X_test: (1417743, 48)
y_test: (1417743,)

RAM-FRIENDLY DATASET READY
Features: 48
X_train: (300000, 48)
y_train: (300000,)
X_test: (1417743, 48)
y_test: (1417743,)
X_train dtype: float32
X_test dtype: float32
X_train RAM: 54.93 MB
X_test RAM: 259.6 MB


In [ ]:

features_23 = [
    'speed',
    'lag_1',
    'rolling_mean_3',
    'lag_2',
    'rolling_mean_6',
    'lag_3',
    'lag_4',
    'rolling_mean_12',
    'lag_6',
    'minute',
    'speed_change_3',
    'speed_change_1',
    'hour_cos',
    'temp_humidity_interaction',
    'hour',
    'speed_vs_mean_6',
    'temperature_2m',
    'speed_change_6',
    'relative_humidity_2m',
    'rolling_mean_24',
    'lag_9',
    'hour_sin',
    'lag_12'
]

print('Core-23 feature count:', len(features_23))
print(features_23)


Core-23 feature count: 23
['speed', 'lag_1', 'rolling_mean_3', 'lag_2', 'rolling_mean_6', 'lag_3', 'lag_4', 'rolling_mean_12', 'lag_6', 'minute', 'speed_change_3', 'speed_change_1', 'hour_cos', 'temp_humidity_interaction', 'hour', 'speed_vs_mean_6', 'temperature_2m', 'speed_change_6', 'relative_humidity_2m', 'rolling_mean_24', 'lag_9', 'hour_sin', 'lag_12']


## 6. Spatial Graph

In [ ]:

graph_path = '../../dataset/METR-LA/adj_mx_METR-LA.pkl'

with open(graph_path, 'rb') as f:
    adj_data = pickle.load(f, encoding='latin1')

adj_sensor_ids = list(adj_data[0])
adj_matrix = np.asarray(adj_data[2], dtype=np.float32)

traffic_sensor_ids = [str(x) for x in sensor_columns]
graph_sensor_ids = [str(x) for x in adj_sensor_ids]

if set(traffic_sensor_ids) != set(graph_sensor_ids):
    raise ValueError('Traffic sensors and graph sensors do not match.')

graph_index = {
    sensor_id: i
    for i, sensor_id in enumerate(graph_sensor_ids)
}

reorder = np.array(
    [graph_index[sensor_id] for sensor_id in traffic_sensor_ids],
    dtype=np.int32
)

adj_matrix = adj_matrix[reorder][:, reorder]

adj_no_self = adj_matrix.copy()
np.fill_diagonal(adj_no_self, 0)

row_sum = adj_no_self.sum(axis=1, keepdims=True)
row_sum[row_sum == 0] = 1.0

W = (adj_no_self / row_sum).astype(np.float32)

print('Adjacency:', adj_matrix.shape)
print('Non-zero edges:', np.count_nonzero(adj_no_self))


Adjacency: (207, 207)
Non-zero edges: 1515


In [7]:
# BUILD SPATIAL MATRICES

neighbor_mean = (speed_matrix @ W.T).astype(np.float32)

neighbor_lag1 = np.zeros_like(neighbor_mean, dtype=np.float32)
neighbor_lag1[1:] = neighbor_mean[:-1]

neighbor_lag3 = np.zeros_like(neighbor_mean, dtype=np.float32)
neighbor_lag3[3:] = neighbor_mean[:-3]

neighbor_change = (
    neighbor_mean - neighbor_lag1
).astype(np.float32)

neighbor_mask = (adj_no_self > 0).astype(np.float32)
neighbor_count = neighbor_mask.sum(axis=1)
neighbor_count[neighbor_count == 0] = 1.0

neighbor_sum = speed_matrix @ neighbor_mask.T
neighbor_sum_sq = (speed_matrix ** 2) @ neighbor_mask.T

neighbor_mean_unweighted = neighbor_sum / neighbor_count
neighbor_var = (
    neighbor_sum_sq / neighbor_count
    - neighbor_mean_unweighted ** 2
)

neighbor_var = np.maximum(neighbor_var, 0)
neighbor_std = np.sqrt(neighbor_var).astype(np.float32)

del neighbor_sum, neighbor_sum_sq
del neighbor_mean_unweighted, neighbor_var, neighbor_mask
gc.collect()

print('neighbor_mean:', neighbor_mean.shape)
print('neighbor_std:', neighbor_std.shape)


neighbor_mean: (34272, 207)
neighbor_std: (34272, 207)


## 7. Exact Pair Feature Builder

In [ ]:

def build_direct_features(
    times,
    sensors,
    selected_features,
    include_spatial=True
):

    n = len(times)

    X = np.empty(
        (
            n,
            len(selected_features)
        ),
        dtype=np.float32
    )

    feature_to_col = {
        name: i
        for i, name in enumerate(
            selected_features
        )
    }


    if 'sensor_id_encoded' in feature_to_col:

        X[:, feature_to_col[
            'sensor_id_encoded'
        ]] = sensor_encoded[
            sensors
        ]


    if 'speed' in feature_to_col:

        X[:, feature_to_col[
            'speed'
        ]] = speed_matrix[
            times,
            sensors
        ]


    
    lag_map = {
        'lag_1': 1,
        'lag_2': 2,
        'lag_3': 3,
        'lag_4': 4,
        'lag_6': 6,
        'lag_9': 9,
        'lag_12': 12,
        'lag_18': 18,
        'lag_24': 24
    }


    for feature, lag in lag_map.items():

        if feature in feature_to_col:

            X[:, feature_to_col[
                feature
            ]] = speed_matrix[
                times - lag,
                sensors
            ]


    rolling_cache = {}


    for window in [
        3,
        6,
        12,
        24
    ]:

        needed = (
            f'rolling_mean_{window}'
            in feature_to_col
        )

        if needed:

            values = np.empty(
                n,
                dtype=np.float32
            )

            for i in range(n):

                tt = times[i]
                ss = sensors[i]

                values[i] = np.mean(
                    speed_matrix[
                        tt - window:tt,
                        ss
                    ]
                )

            rolling_cache[
                f'mean_{window}'
            ] = values


            X[:, feature_to_col[
                f'rolling_mean_{window}'
            ]] = values


    for window in [
        6,
        12,
        24
    ]:

        feature = (
            f'rolling_std_{window}'
        )

        if feature in feature_to_col:

            values = np.empty(
                n,
                dtype=np.float32
            )

            for i in range(n):

                tt = times[i]
                ss = sensors[i]

                values[i] = np.std(
                    speed_matrix[
                        tt - window:tt,
                        ss
                    ]
                )

            X[:, feature_to_col[
                feature
            ]] = values


    change_map = {
        'speed_change_1': 1,
        'speed_change_3': 3,
        'speed_change_6': 6
    }


    current = speed_matrix[
        times,
        sensors
    ]


    for feature, lag in change_map.items():

        if feature in feature_to_col:

            X[:, feature_to_col[
                feature
            ]] = (
                current
                -
                speed_matrix[
                    times - lag,
                    sensors
                ]
            )


   
    for window in [
        6,
        12
    ]:

        feature = (
            f'speed_vs_mean_{window}'
        )

        if feature in feature_to_col:

            if (
                f'mean_{window}'
                in rolling_cache
            ):

                mean_value = rolling_cache[
                    f'mean_{window}'
                ]

            else:

                mean_value = np.empty(
                    n,
                    dtype=np.float32
                )

                for i in range(n):

                    tt = times[i]
                    ss = sensors[i]

                    mean_value[i] = np.mean(
                        speed_matrix[
                            tt-window:tt,
                            ss
                        ]
                    )

            X[:, feature_to_col[
                feature
            ]] = (
                current
                /
                (
                    mean_value
                    + 1e-6
                )
            )


    
    weather_map = {
        'temperature_2m': temperature,
        'relative_humidity_2m': humidity,
        'rain': rain,
        'weather_code': weather_code,
        'temp_humidity_interaction':
            temp_humidity,
        'rain_flag': rain_flag,
        'rain_intensity': rain_intensity,
        'bad_weather': bad_weather
    }


    for feature, values in weather_map.items():

        if feature in feature_to_col:

            X[:, feature_to_col[
                feature
            ]] = values[
                times
            ]


    time_map = {
        'hour': hour,
        'minute': minute,
        'day_of_week': dow,
        'month': month,

        'is_weekend': is_weekend,
        'is_holiday': is_holiday,

        'is_morning_rush':
            morning_rush,

        'is_evening_rush':
            evening_rush,

        'is_rush_hour':
            rush_hour,

        'hour_sin': hour_sin,
        'hour_cos': hour_cos,

        'dow_sin': dow_sin,
        'dow_cos': dow_cos,

        'month_sin': month_sin,
        'month_cos': month_cos
    }


    for feature, values in time_map.items():

        if feature in feature_to_col:

            X[:, feature_to_col[
                feature
            ]] = values[
                times
            ]



    if 'event_count' in feature_to_col:

        X[:, feature_to_col[
            'event_count'
        ]] = event_count_time[
            times
        ]


    if 'has_event' in feature_to_col:

        X[:, feature_to_col[
            'has_event'
        ]] = has_event_time[
            times
        ]


    
    if include_spatial:

        spatial_map = {

            'neighbor_mean':
                neighbor_mean,

            'neighbor_lag1':
                neighbor_lag1,

            'neighbor_lag3':
                neighbor_lag3,

            'neighbor_change':
                neighbor_change,

            'neighbor_std':
                neighbor_std
        }


        for feature, values in spatial_map.items():

            if feature in feature_to_col:

                X[:, feature_to_col[
                    feature
                ]] = values[
                    times,
                    sensors
                ]


    return X

In [ ]:

TRAIN_SAMPLES = 300_000
rng = np.random.default_rng(42)

t_train = rng.choice(
    train_time_positions,
    size=TRAIN_SAMPLES,
    replace=True
)

s_train = rng.integers(
    0,
    N_SENSORS,
    size=TRAIN_SAMPLES
)

spatial_feature_names = (
    features_23
    + [
        'neighbor_mean',
        'neighbor_lag1',
        'neighbor_lag3',
        'neighbor_change',
        'neighbor_std'
    ]
)

X_spatial_train = build_direct_features(
    t_train,
    s_train,
    spatial_feature_names,
    include_spatial=True
)

y_spatial_train = speed_matrix[
    t_train + 3,
    s_train
].astype(np.float32)

print('X_spatial_train:', X_spatial_train.shape)
print('y_spatial_train:', y_spatial_train.shape)
print('RAM:', round(X_spatial_train.nbytes / 1024**2, 2), 'MB')


X_spatial_train: (300000, 28)
y_spatial_train: (300000,)
RAM: 32.04 MB


In [ ]:

test_times = np.repeat(test_time_positions, N_SENSORS)
test_sensors = np.tile(
    np.arange(N_SENSORS, dtype=np.int16),
    len(test_time_positions)
)

TEST_BATCH_SIZE = 50_000
test_batches = []

for start in range(0, len(test_times), TEST_BATCH_SIZE):
    end = min(start + TEST_BATCH_SIZE, len(test_times))

    X_batch = build_direct_features(
        test_times[start:end],
        test_sensors[start:end],
        spatial_feature_names,
        include_spatial=True
    )

    test_batches.append(X_batch)
    print(f'Built {end:,} / {len(test_times):,}')

X_spatial_test = np.concatenate(
    test_batches,
    axis=0
).astype(np.float32, copy=False)

y_spatial_test = speed_matrix[
    test_times + 3,
    test_sensors
].astype(np.float32)

del test_batches, test_times, test_sensors
gc.collect()

print('X_spatial_test:', X_spatial_test.shape)
print('y_spatial_test:', y_spatial_test.shape)
print('RAM:', round(X_spatial_test.nbytes / 1024**2, 2), 'MB')
print('NaN:', np.isnan(X_spatial_test).sum())
print('Inf:', np.isinf(X_spatial_test).sum())


Built 50,000 / 1,417,743
Built 100,000 / 1,417,743
Built 150,000 / 1,417,743
Built 200,000 / 1,417,743
Built 250,000 / 1,417,743
Built 300,000 / 1,417,743
Built 350,000 / 1,417,743
Built 400,000 / 1,417,743
Built 450,000 / 1,417,743
Built 500,000 / 1,417,743
Built 550,000 / 1,417,743
Built 600,000 / 1,417,743
Built 650,000 / 1,417,743
Built 700,000 / 1,417,743
Built 750,000 / 1,417,743
Built 800,000 / 1,417,743
Built 850,000 / 1,417,743
Built 900,000 / 1,417,743
Built 950,000 / 1,417,743
Built 1,000,000 / 1,417,743
Built 1,050,000 / 1,417,743
Built 1,100,000 / 1,417,743
Built 1,150,000 / 1,417,743
Built 1,200,000 / 1,417,743
Built 1,250,000 / 1,417,743
Built 1,300,000 / 1,417,743
Built 1,350,000 / 1,417,743
Built 1,400,000 / 1,417,743
Built 1,417,743 / 1,417,743
X_spatial_test: (1417743, 28)
y_spatial_test: (1417743,)
RAM: 151.43 MB
NaN: 0
Inf: 0


## 10. Spatial Extra Trees

In [11]:
spatial_model = ExtraTreesRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=3,
    max_features=0.7,
    random_state=42,
    n_jobs=1
)

spatial_model.fit(
    X_spatial_train,
    y_spatial_train
)

y_pred_spatial_parts = []

for start in range(0, len(X_spatial_test), 50_000):
    end = min(start + 50_000, len(X_spatial_test))

    pred = spatial_model.predict(
        X_spatial_test[start:end]
    )

    y_pred_spatial_parts.append(
        pred.astype(np.float32)
    )

    print(f'Predicted {end:,} / {len(X_spatial_test):,}')

y_pred_spatial = np.concatenate(y_pred_spatial_parts)
del y_pred_spatial_parts
gc.collect()

spatial_mae = mean_absolute_error(y_spatial_test, y_pred_spatial)
spatial_rmse = np.sqrt(mean_squared_error(y_spatial_test, y_pred_spatial))
spatial_r2 = r2_score(y_spatial_test, y_pred_spatial)

print('=' * 60)
print('SPATIAL EXTRA TREES')
print(f'MAE : {spatial_mae:.4f}')
print(f'RMSE: {spatial_rmse:.4f}')
print(f'R²  : {spatial_r2:.4f}')


Predicted 50,000 / 1,417,743
Predicted 100,000 / 1,417,743
Predicted 150,000 / 1,417,743
Predicted 200,000 / 1,417,743
Predicted 250,000 / 1,417,743
Predicted 300,000 / 1,417,743
Predicted 350,000 / 1,417,743
Predicted 400,000 / 1,417,743
Predicted 450,000 / 1,417,743
Predicted 500,000 / 1,417,743
Predicted 550,000 / 1,417,743
Predicted 600,000 / 1,417,743
Predicted 650,000 / 1,417,743
Predicted 700,000 / 1,417,743
Predicted 750,000 / 1,417,743
Predicted 800,000 / 1,417,743
Predicted 850,000 / 1,417,743
Predicted 900,000 / 1,417,743
Predicted 950,000 / 1,417,743
Predicted 1,000,000 / 1,417,743
Predicted 1,050,000 / 1,417,743
Predicted 1,100,000 / 1,417,743
Predicted 1,150,000 / 1,417,743
Predicted 1,200,000 / 1,417,743
Predicted 1,250,000 / 1,417,743
Predicted 1,300,000 / 1,417,743
Predicted 1,350,000 / 1,417,743
Predicted 1,400,000 / 1,417,743
Predicted 1,417,743 / 1,417,743
SPATIAL EXTRA TREES
MAE : 4.1946
RMSE: 9.2650
R²  : 0.8345


## 11. HistGradientBoosting

In [12]:
hgb_model = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    max_depth=None,
    min_samples_leaf=30,
    l2_regularization=1.0,
    random_state=42
)

hgb_model.fit(
    X_spatial_train,
    y_spatial_train
)

y_pred_hgb_parts = []

for start in range(0, len(X_spatial_test), 50_000):
    end = min(start + 50_000, len(X_spatial_test))

    pred = hgb_model.predict(
        X_spatial_test[start:end]
    )

    y_pred_hgb_parts.append(
        pred.astype(np.float32)
    )

    print(f'Predicted {end:,} / {len(X_spatial_test):,}')

y_pred_hgb = np.concatenate(y_pred_hgb_parts)
del y_pred_hgb_parts
gc.collect()

hgb_mae = mean_absolute_error(y_spatial_test, y_pred_hgb)
hgb_rmse = np.sqrt(mean_squared_error(y_spatial_test, y_pred_hgb))
hgb_r2 = r2_score(y_spatial_test, y_pred_hgb)

print('=' * 60)
print('HGB')
print(f'MAE : {hgb_mae:.4f}')
print(f'RMSE: {hgb_rmse:.4f}')
print(f'R²  : {hgb_r2:.4f}')


Predicted 50,000 / 1,417,743
Predicted 100,000 / 1,417,743
Predicted 150,000 / 1,417,743
Predicted 200,000 / 1,417,743
Predicted 250,000 / 1,417,743
Predicted 300,000 / 1,417,743
Predicted 350,000 / 1,417,743
Predicted 400,000 / 1,417,743
Predicted 450,000 / 1,417,743
Predicted 500,000 / 1,417,743
Predicted 550,000 / 1,417,743
Predicted 600,000 / 1,417,743
Predicted 650,000 / 1,417,743
Predicted 700,000 / 1,417,743
Predicted 750,000 / 1,417,743
Predicted 800,000 / 1,417,743
Predicted 850,000 / 1,417,743
Predicted 900,000 / 1,417,743
Predicted 950,000 / 1,417,743
Predicted 1,000,000 / 1,417,743
Predicted 1,050,000 / 1,417,743
Predicted 1,100,000 / 1,417,743
Predicted 1,150,000 / 1,417,743
Predicted 1,200,000 / 1,417,743
Predicted 1,250,000 / 1,417,743
Predicted 1,300,000 / 1,417,743
Predicted 1,350,000 / 1,417,743
Predicted 1,400,000 / 1,417,743
Predicted 1,417,743 / 1,417,743
HGB
MAE : 4.3662
RMSE: 9.3228
R²  : 0.8325


## 12. Validation-Based Ensemble Weight

In [13]:
VAL_SAMPLES = 100_000
rng = np.random.default_rng(2026)

val_sample_times = rng.choice(
    valid_times,
    size=VAL_SAMPLES,
    replace=True
)

val_sample_sensors = rng.integers(
    0,
    N_SENSORS,
    size=VAL_SAMPLES
)

X_val_ensemble = build_direct_features(
    val_sample_times,
    val_sample_sensors,
    spatial_feature_names,
    include_spatial=True
)

y_val_ensemble = speed_matrix[
    val_sample_times + 3,
    val_sample_sensors
].astype(np.float32)

val_pred_extra = spatial_model.predict(
    X_val_ensemble
).astype(np.float32)

val_pred_hgb = hgb_model.predict(
    X_val_ensemble
).astype(np.float32)

best_weight = None
best_r2 = -np.inf

for w in np.arange(0.0, 1.01, 0.05):
    pred = (
        w * val_pred_extra
        +
        (1.0 - w) * val_pred_hgb
    )

    r2 = r2_score(y_val_ensemble, pred)

    if r2 > best_r2:
        best_r2 = r2
        best_weight = w

best_hgb_weight = 1.0 - best_weight

best_pred = (
    best_weight * val_pred_extra
    +
    best_hgb_weight * val_pred_hgb
)

best_mae = mean_absolute_error(y_val_ensemble, best_pred)
best_rmse = np.sqrt(mean_squared_error(y_val_ensemble, best_pred))

print('=' * 60)
print('BEST ENSEMBLE')
print(f'Extra Trees weight: {best_weight:.2f}')
print(f'HGB weight:         {best_hgb_weight:.2f}')
print(f'MAE : {best_mae:.4f}')
print(f'RMSE: {best_rmse:.4f}')
print(f'R²  : {best_r2:.4f}')

del X_val_ensemble, y_val_ensemble
del val_pred_extra, val_pred_hgb, best_pred
del val_sample_times, val_sample_sensors
gc.collect()


BEST ENSEMBLE
Extra Trees weight: 0.50
HGB weight:         0.50
MAE : 3.6900
RMSE: 7.7917
R²  : 0.8522


23

## 13. Final Test Result

In [14]:
# The validation search selected 50/50.
FINAL_ET_WEIGHT = 0.50
FINAL_HGB_WEIGHT = 0.50

final_ensemble_pred = (
    FINAL_ET_WEIGHT * y_pred_spatial
    +
    FINAL_HGB_WEIGHT * y_pred_hgb
).astype(np.float32)

final_mae = mean_absolute_error(
    y_spatial_test,
    final_ensemble_pred
)

final_rmse = np.sqrt(
    mean_squared_error(
        y_spatial_test,
        final_ensemble_pred
    )
)

final_r2 = r2_score(
    y_spatial_test,
    final_ensemble_pred
)

print('=' * 65)
print('FINAL MODEL RESULTS')
print('=' * 65)
print('Core-23 baseline      R² = 0.8304')
print(f'Spatial Extra Trees   R² = {spatial_r2:.4f}')
print(f'HGB                   R² = {hgb_r2:.4f}')
print(f'50/50 Ensemble        R² = {final_r2:.4f}')
print()
print(f'Final MAE : {final_mae:.4f}')
print(f'Final RMSE: {final_rmse:.4f}')
print(f'Final R²  : {final_r2:.4f}')
print('=' * 65)


FINAL MODEL RESULTS
Core-23 baseline      R² = 0.8304
Spatial Extra Trees   R² = 0.8345
HGB                   R² = 0.8325
50/50 Ensemble        R² = 0.8356

Final MAE : 4.2584
Final RMSE: 9.2358
Final R²  : 0.8356


In [ ]:

# Keep:
#   spatial_model
#   hgb_model
#   final_ensemble_pred
#
# Release large arrays when no longer needed:
#
# del X_spatial_train, y_spatial_train
# del X_spatial_test, y_spatial_test
# del y_pred_spatial, y_pred_hgb
# gc.collect()

print('Final pipeline complete.')


Final pipeline complete.


In [ ]:
import joblib
import json
from pathlib import Path

MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(
    spatial_model,
    MODEL_DIR / 'extra_trees_spatial.joblib'
)

joblib.dump(
    hgb_model,
    MODEL_DIR / 'hgb_spatial.joblib'
)

config = {
    'features': spatial_feature_names,
    'extra_trees_weight': 0.50,
    'hgb_weight': 0.50,
    'prediction_horizon_minutes': 15,
    'sensor_count': 207,
    'test_r2': 0.8358
}

with open(
    MODEL_DIR / 'config.json',
    'w',
    encoding='utf-8'
) as f:
    json.dump(
        config,
        f,
        indent=4,
        ensure_ascii=False
    )

print("=" * 60)
print("FINAL MODELS SAVED")
print("=" * 60)

print(
    "Extra Trees:",
    MODEL_DIR / "extra_trees_spatial.joblib"
)

print(
    "HGB:",
    MODEL_DIR / "hgb_spatial.joblib"
)

print(
    "Config:",
    MODEL_DIR / "config.json"
)